# Lab4.2 — Conversie color → grayscale

Conversia la grayscale reduce fiecare pixel de la trei canale la un singur canal. Este una dintre cele mai frecvente etape de preprocesare în computer vision.


## Obiective

- conversia BGR → grayscale;
- implementarea unei conversii cu NumPy;
- compararea rezultatului cu OpenCV;
- observarea reducerii volumului de date;
- măsurarea timpului de execuție în software.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

IMAGE_DIR = Path("images")
IMAGE_DIR.mkdir(exist_ok=True)
IMAGE_PATH = IMAGE_DIR / "opencv_filters.jpg"

IMAGE_URL = (
    "https://raw.githubusercontent.com/onigas/PYNQ/master/"
    "boards/Pynq-Z1/base/notebooks/video/data/opencv_filters.jpg"
)

if not IMAGE_PATH.exists():
    print("Downloading test image ...")
    try:
        urlretrieve(IMAGE_URL, IMAGE_PATH)
    except Exception as e:
        raise RuntimeError(
            "Imaginea nu a putut fi descărcată. "
            "Copiați manual opencv_filters.jpg în Lab4/images/."
        ) from e

print("Image:", IMAGE_PATH)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time

image_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
if image_bgr is None:
    raise RuntimeError("Imaginea nu a putut fi încărcată.")

## 1. Conversia cu OpenCV


In [ ]:
gray_cv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

print("Color shape:", image_bgr.shape)
print("Gray shape :", gray_cv.shape)
print("Color bytes:", image_bgr.nbytes)
print("Gray bytes :", gray_cv.nbytes)

In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(gray_cv, cmap="gray")
plt.title("Grayscale")
plt.axis("off")
plt.show()

## 2. Conversia cu NumPy

O aproximare uzuală a luminanței este:

```text
Y = 0.299 R + 0.587 G + 0.114 B
```


In [ ]:
b = image_bgr[:, :, 0].astype(np.float32)
g = image_bgr[:, :, 1].astype(np.float32)
r = image_bgr[:, :, 2].astype(np.float32)

gray_np = 0.114 * b + 0.587 * g + 0.299 * r
gray_np = np.clip(gray_np, 0, 255).astype(np.uint8)

print("NumPy gray shape:", gray_np.shape)
print("NumPy gray dtype:", gray_np.dtype)

## 3. Compararea rezultatelor

Implementările pot diferi ușor din cauza rotunjirii și a optimizărilor interne OpenCV.


In [ ]:
difference = cv2.absdiff(gray_cv, gray_np)

print("Maximum difference:", int(difference.max()))
print("Mean difference   :", float(difference.mean()))

## 4. Timpul de execuție

Executăm fiecare operație de mai multe ori. Prima execuție este făcută separat ca warm-up.


In [ ]:
REPEATS = 50

cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

t0 = time.perf_counter()
for _ in range(REPEATS):
    tmp = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
t_cv = (time.perf_counter() - t0) / REPEATS

t0 = time.perf_counter()
for _ in range(REPEATS):
    b = image_bgr[:, :, 0].astype(np.float32)
    g = image_bgr[:, :, 1].astype(np.float32)
    r = image_bgr[:, :, 2].astype(np.float32)
    tmp = np.clip(0.114*b + 0.587*g + 0.299*r, 0, 255).astype(np.uint8)
t_np = (time.perf_counter() - t0) / REPEATS

print(f"OpenCV: {t_cv*1e3:.3f} ms")
print(f"NumPy : {t_np*1e3:.3f} ms")

OpenCV execută operațiile numerice în cod nativ optimizat. NumPy execută de asemenea operații vectorizate în cod nativ, dar expresia NumPy creează mai multe tablouri temporare. Ambele variante rulează pe **PS/ARM**, nu în FPGA.


## 5. Exercițiu

Calculați pentru cele două variante:

- timpul mediu în ms;
- FPS teoretic `1 / timp`;
- raportul de viteză dintre implementări.


In [ ]:
# Scrieți aici soluția


## 6. Întrebări

1. De ce imaginea grayscale ocupă aproximativ de trei ori mai puțină memorie decât imaginea color?
2. De ce rezultatul NumPy poate diferi cu 1–2 niveluri de gri față de OpenCV?
3. De ce vectorizarea este preferabilă unei bucle Python pixel-cu-pixel?
